In [ ]:
# -*- coding: utf-8 -*-
"""
YouTube Channel Transcript Downloader (Safe + Scheduled + Resumable)

Changes in this version:
- Saves each transcript as: YYYYMMDD_<video_id>.txt
- Uses the video's YouTube upload date before the identifier.
- If an older file named <video_id>.txt already exists, it renames it to the
  new date-based filename instead of downloading the transcript again.

Features:
- Skips already downloaded files
- Runs repeatedly on a schedule
- Uses safe parallelism to avoid blocking
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable,
)


# ================= CONFIG =================
CHANNEL_HANDLE = "@FelixFriends"
OUTPUT_DIR = Path("transcripts_ids")

MAX_PER_RUN = 4           # videos per run
MAX_WORKERS = 1            # parallel threads; keep low
DELAY_RANGE = (25, 32)     # seconds between transcript requests
RUN_INTERVAL_HOURS = .15     # repeat interval
# ==========================================


@dataclass(frozen=True)
class VideoInfo:
    video_id: str
    upload_date: str | None = None   # normalized as YYYYMMDD when available


_DATE_YYYYMMDD = re.compile(r"^\d{8}$")
_DATE_YYYY_MM_DD = re.compile(r"^\d{4}-\d{2}-\d{2}$")


def normalize_upload_date(value) -> str | None:
    """Return a date string as YYYYMMDD, or None if unavailable."""
    if value is None:
        return None

    # yt-dlp usually returns upload_date as 'YYYYMMDD'.
    if isinstance(value, str):
        value = value.strip()
        if _DATE_YYYYMMDD.match(value):
            return value
        if _DATE_YYYY_MM_DD.match(value):
            return value.replace("-", "")

        # Some extractors may return an ISO-like datetime string.
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00")).date().strftime("%Y%m%d")
        except ValueError:
            return None

    # Some metadata fields use Unix timestamps.
    if isinstance(value, (int, float)):
        try:
            return datetime.fromtimestamp(value, timezone.utc).date().strftime("%Y%m%d")
        except (OverflowError, OSError, ValueError):
            return None

    return None


def get_video_entries(channel_handle: str) -> list[VideoInfo]:
    """Fetch video IDs and whatever upload-date metadata yt-dlp provides."""
    url = f"https://www.youtube.com/{channel_handle}/videos"

    ydl_opts = {
        "quiet": True,
        "extract_flat": True,
        "skip_download": True,
        "ignoreerrors": True,
    }

    video_entries: list[VideoInfo] = []

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

        for entry in info.get("entries", []):
            if not entry or "id" not in entry:
                continue

            raw_date = (
                entry.get("upload_date")
                or entry.get("release_date")
                or entry.get("timestamp")
                or entry.get("release_timestamp")
            )

            video_entries.append(
                VideoInfo(
                    video_id=entry["id"],
                    upload_date=normalize_upload_date(raw_date),
                )
            )

    return video_entries


def fetch_upload_date(video_id: str) -> str | None:
    """Fetch full video metadata when the flat channel listing lacks upload_date."""
    url = f"https://www.youtube.com/watch?v={video_id}"

    ydl_opts = {
        "quiet": True,
        "skip_download": True,
        "noplaylist": True,
        "ignoreerrors": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    if not info:
        return None

    raw_date = (
        info.get("upload_date")
        or info.get("release_date")
        or info.get("timestamp")
        or info.get("release_timestamp")
    )

    return normalize_upload_date(raw_date)


def fetch_transcript(video_id: str):
    """Fetch transcript safely."""
    api = YouTubeTranscriptApi()
    return api.fetch(video_id, languages=["en"])


def transcript_to_text(transcript) -> str:
    return "\n".join(
        segment.text.strip()
        for segment in transcript
        if segment.text and segment.text.strip()
    )


def transcript_file(video_id: str, upload_date: str) -> Path:
    """New filename format: upload date followed by identifier."""
    return OUTPUT_DIR / f"{upload_date}_{video_id}.txt"


def legacy_transcript_file(video_id: str) -> Path:
    """Old filename format used by the original script."""
    return OUTPUT_DIR / f"{video_id}.txt"


def previous_date_style_file(video_id: str, upload_date: str) -> Path:
    """Previous date-prefixed format: YYYY-MM-DD_<video_id>.txt."""
    if _DATE_YYYYMMDD.match(upload_date):
        dashed = f"{upload_date[0:4]}-{upload_date[4:6]}-{upload_date[6:8]}"
        return OUTPUT_DIR / f"{dashed}_{video_id}.txt"
    return OUTPUT_DIR / f"{upload_date}_{video_id}.txt"


def new_style_file_exists(video_id: str) -> bool:
    """Check if this video already has a date-prefixed transcript file."""
    if not OUTPUT_DIR.exists():
        return False
    return any(OUTPUT_DIR.glob(f"????????_{video_id}.txt")) or any(
        OUTPUT_DIR.glob(f"unknown-date_{video_id}.txt")
    )


def save_transcript(video_id: str, upload_date: str, text: str) -> Path:
    OUTPUT_DIR.mkdir(exist_ok=True)
    path = transcript_file(video_id, upload_date)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path


def process_video(video: VideoInfo) -> str:
    """Process a single video with delay, metadata lookup, and error handling."""
    video_id = video.video_id

    if new_style_file_exists(video_id):
        return f"Skipped (already exists): {video_id}"

    try:
        upload_date = video.upload_date or fetch_upload_date(video_id) or "unknown-date"
        target_path = transcript_file(video_id, upload_date)

        # If a transcript exists under an older filename, rename it instead of
        # downloading the transcript again.
        legacy_path = legacy_transcript_file(video_id)
        previous_date_path = previous_date_style_file(video_id, upload_date)

        if previous_date_path.exists() and not target_path.exists():
            OUTPUT_DIR.mkdir(exist_ok=True)
            previous_date_path.rename(target_path)
            return f"Renamed existing file: {target_path.name}"

        if legacy_path.exists() and not target_path.exists():
            OUTPUT_DIR.mkdir(exist_ok=True)
            legacy_path.rename(target_path)
            return f"Renamed existing file: {target_path.name}"

        if target_path.exists():
            return f"Skipped (already exists): {target_path.name}"

        # Random delay BEFORE transcript request.
        time.sleep(random.uniform(*DELAY_RANGE))

        transcript = fetch_transcript(video_id)
        text = transcript_to_text(transcript)
        saved_path = save_transcript(video_id, upload_date, text)

        return f"Saved: {saved_path.name}"

    except TranscriptsDisabled:
        return f"Skipped (disabled): {video_id}"
    except NoTranscriptFound:
        return f"Skipped (no transcript): {video_id}"
    except VideoUnavailable:
        return f"Skipped (unavailable): {video_id}"
    except Exception as e:
        return f"Error ({video_id}): {e}"


def run_batch(videos: list[VideoInfo]) -> None:
    """Run a safe parallel batch."""
    print(f"Processing {len(videos)} videos...\n")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_video, video): video.video_id for video in videos}

        for future in as_completed(futures):
            print(future.result())


def main_loop() -> None:
    """Runs forever, executing on the configured interval."""
    while True:
        print("\n=== NEW RUN STARTED ===\n")

        try:
            all_videos = get_video_entries(CHANNEL_HANDLE)

            # New-style files are considered complete.
            # Older files are intentionally not filtered here because the
            # program can rename them into the new format during processing.
            remaining = [video for video in all_videos if not new_style_file_exists(video.video_id)]

            print(f"Total videos: {len(all_videos)}")
            print(f"Remaining: {len(remaining)}")

            if not remaining:
                print("All transcripts downloaded!")
            else:
                batch = remaining[:MAX_PER_RUN]
                run_batch(batch)

        except Exception as e:
            print(f"Fatal error: {e}")

        print(f"\nSleeping for {RUN_INTERVAL_HOURS} hours...\n")
        time.sleep(RUN_INTERVAL_HOURS * 3600)


if __name__ == "__main__":
    main_loop()



=== NEW RUN STARTED ===

Total videos: 1161
Remaining: 1161
Processing 4 videos...



Saved: 20260520_0fTxc_eCXek.txt


Saved: 20260518_C3Fj2o9mBHI.txt


Saved: 20260517_ymJm0MRecV8.txt


Saved: 20260515_6TRPy8UWvPQ.txt

Sleeping for 0.2 hours...


=== NEW RUN STARTED ===

Total videos: 1161
Remaining: 1157
Processing 4 videos...



...

Saved: 20240414_9KQpsETZA2U.txt

Sleeping for 0.2 hours...


=== NEW RUN STARTED ===

Total videos: 1162
Remaining: 412
Processing 4 videos...



Saved: 20240414_w6XNRvhFi14.txt


Saved: 20240413_TrUav6bMiE8.txt


Saved: 20240413_f_mRcR0ppjs.txt


Saved: 20240412_pltOTz-APNY.txt

Sleeping for 0.2 hours...

